# breast cancer diagnosis

sklearn breast cancer (binary, 569 rows). good for ROC and recall focus.

In [1]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

bc = load_breast_cancer()
X = pd.DataFrame(bc.data, columns=bc.feature_names)
y = bc.target  # 0 = malignant, 1 = benign
X.shape

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
pipe = Pipeline([('scale', StandardScaler()), ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
pipe.fit(Xtr, ytr)
pipe.score(Xte, yte)

## ROC curve

In [3]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
%matplotlib inline
probs = pipe.predict_proba(Xte)[:, 1]
fpr, tpr, _ = roc_curve(yte, probs)
plt.plot(fpr, tpr, label='AUC=%.3f' % auc(fpr, tpr))
plt.plot([0,1],[0,1], 'k--')
plt.legend()
plt.show()

## bake-off

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import xgboost as xgb
models = {
    'lr+sc': pipe,
    'rf':    Pipeline([('m', RandomForestClassifier(n_estimators=200, random_state=0))]),
    'svm':   Pipeline([('s', StandardScaler()), ('m', SVC(gamma='scale'))]),
    'xgb':   Pipeline([('m', xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=0,
                                              use_label_encoder=False, eval_metric='logloss'))]),
}
import pandas as pd
out = {n: cross_val_score(m, X, y, cv=5).mean() for n, m in models.items()}
pd.Series(out).sort_values(ascending=False)

## RF feature importance

In [5]:
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X, y)
import pandas as pd
pd.Series(rf.feature_importances_, index=load_breast_cancer().feature_names).sort_values().tail(10)

## classification report

In [6]:
from sklearn.metrics import classification_report
print(classification_report(yte, pipe.predict(Xte), target_names=['malignant','benign']))

In [ ]:
# saved

hp tweak.